# 13 - Fetch forecast wind (Open-Meteo) for v3

Pulls hourly 10 m wind for each of the 37 stations over the full dataset range,
from two Open-Meteo archives:

| source | endpoint | meaning |
|---|---|---|
| **reanalysis** | `archive-api` (ERA5) | what the wind *actually did* - the perfect-forecast upper bound |
| **forecast**   | `historical-forecast-api` | what the wind forecast *said* at the time - the realistic version |

Both free, no API key. Wind is converted to u/v with the same meteorological
convention as `01_preprocessing` (`u = -WS*sin(WD)`, `v = -WS*cos(WD)`), aligned
to the exact CPCB hourly timestamp grid, and saved as `(T, N, 2)` arrays.

Output: `openmeteo_reanalysis_uv.npy`, `openmeteo_forecast_uv.npy`  (each ~5 MB).

Run time ~5-10 min (74 API calls, one per station per source, 1 s spacing).

In [1]:
import os, time, json
import numpy as np
import pandas as pd
import requests

BASE   = r'D:\FYP\Eco-Twin'
V2_DIR = os.path.join(BASE, 'v2_clean')
DATASET_CSV    = os.path.join(V2_DIR, 'EcoTwin_Final_Dataset.csv')
STATIONS_EXCEL = os.path.join(V2_DIR, 'Delhi_list_v2.xlsx')

# station list (same 37, same order as everything downstream)
df_st = pd.read_excel(STATIONS_EXCEL)
df_st.columns = df_st.columns.str.strip().str.lower()
df_st['station'] = df_st['station'].str.strip()
station_names = df_st['station'].tolist()
lats = df_st['latitude'].values.astype(float)
lons = df_st['longitude'].values.astype(float)
N = len(station_names)

# exact hourly timestamp grid from the CPCB dataset
_ts = pd.read_csv(DATASET_CSV, parse_dates=['Timestamp'], usecols=['Timestamp'])
timestamps = pd.DatetimeIndex(sorted(_ts['Timestamp'].unique()))
T = len(timestamps)
START = timestamps[0].strftime('%Y-%m-%d')
END   = timestamps[-1].strftime('%Y-%m-%d')

print(f'Stations : {N}')
print(f'Grid     : {T:,} hourly steps  {timestamps[0]}  ->  {timestamps[-1]}')
print(f'Request  : {START}  ->  {END}  (timezone Asia/Kolkata, wind in m/s)')

# HOW TO CHECK: N == 37, T == 19921, dates 2024-01-01 -> 2026-04-10.

Stations : 37
Grid     : 19,921 hourly steps  2024-01-01 00:00:00  ->  2026-04-10 00:00:00
Request  : 2024-01-01  ->  2026-04-10  (timezone Asia/Kolkata, wind in m/s)


In [2]:
SOURCES = {
    'reanalysis': 'https://archive-api.open-meteo.com/v1/archive',
    'forecast':   'https://historical-forecast-api.open-meteo.com/v1/forecast',
}

def fetch_station(url, lat, lon, retries=4):
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': START, 'end_date': END,
        'hourly': 'wind_speed_10m,wind_direction_10m',
        'wind_speed_unit': 'ms',
        'timezone': 'Asia/Kolkata',
    }
    for k in range(retries):
        try:
            r = requests.get(url, params=params, timeout=90)
            if r.status_code == 200:
                h = r.json()['hourly']
                s = pd.DataFrame({
                    'ws': h['wind_speed_10m'],
                    'wd': h['wind_direction_10m'],
                }, index=pd.DatetimeIndex(h['time']))
                return s
            if r.status_code in (429, 503):          # rate limit / busy -> back off
                time.sleep(8 * (k + 1)); continue
            raise RuntimeError(f'HTTP {r.status_code}: {r.text[:200]}')
        except (requests.Timeout, requests.ConnectionError):
            time.sleep(5 * (k + 1))
    raise RuntimeError('failed after retries')

def wd_ws_to_uv(ws, wd):
    # meteorological: wd = direction FROM which wind blows, deg clockwise from N
    wd = np.asarray(wd, float) % 360
    ws = np.asarray(ws, float)
    rad = np.deg2rad(wd)
    return -ws * np.sin(rad), -ws * np.cos(rad)      # u (east), v (north) of air motion

print('fetch helpers ready')

fetch helpers ready


In [3]:
uv = {}          # source -> (T, N, 2) array of [u, v]
raw = {}         # source -> list of per-station DataFrames (for the sanity check)

for src, url in SOURCES.items():
    print(f'\n=== {src}  ({url.split("//")[1].split("/")[0]}) ===')
    arr = np.zeros((T, N, 2), dtype=np.float32)
    frames = []
    t0 = time.time()
    for i, (nm, la, lo) in enumerate(zip(station_names, lats, lons)):
        s = fetch_station(url, la, lo)
        s = s[~s.index.duplicated()].reindex(timestamps)        # align to CPCB grid
        s['ws'] = s['ws'].interpolate(limit_direction='both')
        s['wd'] = s['wd'].interpolate(limit_direction='both')
        u, v = wd_ws_to_uv(s['ws'].values, s['wd'].values)
        arr[:, i, 0], arr[:, i, 1] = u, v
        frames.append(s)
        if (i + 1) % 10 == 0 or i == N - 1:
            print(f'  [{i+1:>2}/{N}]  {nm[:24]:<24}  {time.time()-t0:>4.0f}s')
        time.sleep(1.0)                                          # polite spacing
    uv[src] = arr
    raw[src] = frames
    nan_frac = np.isnan(arr).mean()
    print(f'  done. shape {arr.shape}  NaN fraction {nan_frac:.4f}')

# HOW TO CHECK: NaN fraction should be ~0 (a few edge hours interpolated is fine).
# If a source 404s or errors for every station, that endpoint name may have changed -
# stop and tell me, reanalysis alone is still enough to proceed.


=== reanalysis  (archive-api.open-meteo.com) ===
  [10/37]  Dr. Karni Singh Shooting    23s
  [20/37]  Okhla Phase-2               47s
  [30/37]  Burari Crossing             70s
  [37/37]  Wazirpur                    90s
  done. shape (19921, 37, 2)  NaN fraction 0.0000

=== forecast  (historical-forecast-api.open-meteo.com) ===
  [10/37]  Dr. Karni Singh Shooting    22s
  [20/37]  Okhla Phase-2               45s
  [30/37]  Burari Crossing             71s
  [37/37]  Wazirpur                    91s
  done. shape (19921, 37, 2)  NaN fraction 0.0000


In [4]:
for src, arr in uv.items():
    arr = np.nan_to_num(arr).astype(np.float32)
    path = os.path.join(V2_DIR, f'openmeteo_{src}_uv.npy')
    np.save(path, arr)
    print(f'saved {path}   {arr.shape}   {arr.nbytes/1e6:.1f} MB   '
          f'u range [{arr[...,0].min():.1f}, {arr[...,0].max():.1f}]  '
          f'v range [{arr[...,1].min():.1f}, {arr[...,1].max():.1f}]')

meta = {'stations': station_names, 'start': START, 'end': END,
        'n_timesteps': int(T), 'timezone': 'Asia/Kolkata',
        'sources': list(SOURCES), 'convention': 'u=-ws*sin(wd), v=-ws*cos(wd), wd meteorological'}
json.dump(meta, open(os.path.join(V2_DIR, 'openmeteo_wind_meta.json'), 'w'), indent=2)
print('\nsaved openmeteo_wind_meta.json')

saved D:\FYP\Eco-Twin\v2_clean\openmeteo_reanalysis_uv.npy   (19921, 37, 2)   5.9 MB   u range [-8.3, 9.4]  v range [-10.5, 8.7]
saved D:\FYP\Eco-Twin\v2_clean\openmeteo_forecast_uv.npy   (19921, 37, 2)   5.9 MB   u range [-8.3, 9.4]  v range [-10.5, 8.7]

saved openmeteo_wind_meta.json


In [5]:
# sanity: Open-Meteo wind vs CPCB observed wind (should be broadly correlated, not identical -
# 10 m model grid vs station-height sensor)
import matplotlib
matplotlib.use('Agg'); import matplotlib.pyplot as plt

cpcb = pd.read_csv(DATASET_CSV, parse_dates=['Timestamp'], usecols=['Timestamp', 'Station', 'u_wind', 'v_wind'])
cpcb['Station'] = cpcb['Station'].str.strip()
cu = cpcb.pivot_table(index='Timestamp', columns='Station', values='u_wind').reindex(columns=station_names).values
cv = cpcb.pivot_table(index='Timestamp', columns='Station', values='v_wind').reindex(columns=station_names).values

print(f"{'source':12} {'corr(u)':>9} {'corr(v)':>9} {'RMSE u':>9} {'RMSE v':>9}")
for src, arr in uv.items():
    a = np.nan_to_num(arr)
    m = ~np.isnan(cu)
    ru = np.corrcoef(a[..., 0][m], cu[m])[0, 1]
    rv = np.corrcoef(a[..., 1][m], cv[m])[0, 1]
    eu = np.sqrt(np.nanmean((a[..., 0] - cu) ** 2))
    ev = np.sqrt(np.nanmean((a[..., 1] - cv) ** 2))
    print(f'{src:12} {ru:>9.3f} {rv:>9.3f} {eu:>9.2f} {ev:>9.2f}')

# one station, one week
si, h0 = 0, 24 * 7
fig, ax = plt.subplots(2, 1, figsize=(13, 5), sharex=True)
for k, (lbl, comp) in enumerate([('u_wind', cu), ('v_wind', cv)]):
    ax[k].plot(comp[:h0, si], label='CPCB observed', lw=1.2, color='#222')
    for src, arr in uv.items():
        ax[k].plot(np.nan_to_num(arr)[:h0, si, k], label=f'Open-Meteo {src}', lw=1)
    ax[k].set_ylabel(lbl); ax[k].legend(fontsize=8); ax[k].grid(alpha=.3)
ax[0].set_title(f'{station_names[si]} - first 7 days')
plt.tight_layout()
plt.savefig(os.path.join(V2_DIR, 'ecotwin_outputs', 'plots', 'openmeteo_wind_check.png'), dpi=140, bbox_inches='tight')
plt.show()

# HOW TO CHECK: corr(u), corr(v) should be roughly 0.6-0.9 for both sources.
# reanalysis usually correlates a bit higher than forecast. If corr < 0.3 the
# alignment or convention is off - stop and tell me.

source         corr(u)   corr(v)    RMSE u    RMSE v
reanalysis       0.252     0.040      1.99      1.84
forecast         0.252     0.040      1.99      1.84


C:\Users\jlakk_j7eh5fd\AppData\Local\Temp\ipykernel_3464\3946594407.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
import numpy as np, pandas as pd
raw = pd.read_csv(DATASET_CSV, parse_dates=['Timestamp'], usecols=['Timestamp','Station','ws','wd','u_wind','v_wind'])
raw['Station'] = raw['Station'].str.strip()
cws = raw.pivot_table(index='Timestamp', columns='Station', values='ws').reindex(columns=station_names).values
cwd = raw.pivot_table(index='Timestamp', columns='Station', values='wd').reindex(columns=station_names).values
om = np.nan_to_num(uv['reanalysis'])
om_spd = np.hypot(om[...,0], om[...,1])
om_dir = (np.degrees(np.arctan2(-om[...,0], -om[...,1])) % 360)   # back to met "from" deg

m = ~np.isnan(cws)
print("SPEED  corr:", round(np.corrcoef(om_spd[m], cws[m])[0,1], 3),
      "| CPCB mean", round(np.nanmean(cws),2), "OM mean", round(om_spd.mean(),2))
print("CPCB ws: frac exactly 0 :", round(np.nanmean(cws==0),3),
      "| per-station std range", round(np.nanmin(np.nanstd(cws,0)),3), "-", round(np.nanmax(np.nanstd(cws,0)),3))

# timezone / lag scan on u
cu = raw.pivot_table(index='Timestamp', columns='Station', values='u_wind').reindex(columns=station_names).values
for lag in range(-6, 7):
    a = np.roll(om[...,0], lag, axis=0)
    mm = ~np.isnan(cu)
    print(f"  lag {lag:+d}h  corr(u) {np.corrcoef(a[mm], cu[mm])[0,1]:+.3f}")

# is CPCB using a different direction convention?
for name, uu, vv in [
    ("met from  (-ws sin, -ws cos)", -cws*np.sin(np.deg2rad(cwd)), -cws*np.cos(np.deg2rad(cwd))),
    ("met to    (+ws sin, +ws cos)",  cws*np.sin(np.deg2rad(cwd)),  cws*np.cos(np.deg2rad(cwd))),
    ("math      (ws cos,   ws sin)",  cws*np.cos(np.deg2rad(cwd)),  cws*np.sin(np.deg2rad(cwd))),
]:
    mm = ~np.isnan(uu)
    print(f"  {name}:  corr(u) {np.corrcoef(np.nan_to_num(uu)[mm], om[...,0][mm])[0,1]:+.3f}"
          f"  corr(v) {np.corrcoef(np.nan_to_num(vv)[mm], om[...,1][mm])[0,1]:+.3f}")

SPEED  corr: 0.347 | CPCB mean 1.05 OM mean 2.13
CPCB ws: frac exactly 0 : 0.0 | per-station std range 0.385 - 1.774
  lag -6h  corr(u) +0.210
  lag -5h  corr(u) +0.219
  lag -4h  corr(u) +0.229
  lag -3h  corr(u) +0.238
  lag -2h  corr(u) +0.246
  lag -1h  corr(u) +0.252
  lag +0h  corr(u) +0.252
  lag +1h  corr(u) +0.249
  lag +2h  corr(u) +0.241
  lag +3h  corr(u) +0.232
  lag +4h  corr(u) +0.223
  lag +5h  corr(u) +0.214
  lag +6h  corr(u) +0.206
  met from  (-ws sin, -ws cos):  corr(u) +0.252  corr(v) +0.040
  met to    (+ws sin, +ws cos):  corr(u) -0.252  corr(v) -0.040
  math      (ws cos,   ws sin):  corr(u) +0.023  corr(v) +0.137


In [8]:
om = np.load(os.path.join(V2_DIR, 'openmeteo_reanalysis_uv.npy'))
# spatial spread of wind at each hour
sp = np.hypot(om[...,0].std(1), om[...,1].std(1))          # std across 37 stations, per hour
print("cross-station wind spread: mean", round(sp.mean(),2), "m/s  p90", round(np.percentile(sp,90),2))
u = np.unique(np.round(om.reshape(-1,74),1), axis=0)       # rough distinct wind vectors
print("distinct station wind profiles (approx):", (np.abs(om[1000] - om[1000,0]).sum(1) > 0.1).sum(), "/ 37 differ from station 0 at one hour")

cross-station wind spread: mean 0.47 m/s  p90 0.79
distinct station wind profiles (approx): 31 / 37 differ from station 0 at one hour
